# Step 4: Elevation data and terrain layers (slope, aspect, curvature)

**What this notebook does**
1. Works out which 1 degree elevation tiles cover our 10 km corridor
2. Downloads them from the public Copernicus DEM (30 m) on AWS (no account needed)
3. Reprojects to metres (UTM 43N) and cuts to the corridor
4. Computes slope, aspect and a simple curvature
5. Checks the result against the landslide points: landslides should sit on steeper ground than the average pixel

Needs: `data/processed/corridor_10km.geojson` (Step 2) and, for the check, `data/processed/landslides_su2022_in_corridor.geojson` (Step 3b).

This step uses `rasterio`. If `rasterio` is blocked on your laptop, run this notebook in Google Colab: Cell 0 sets everything up there (it does nothing on your own computer).

Data: Copernicus DEM GLO-30, © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA. It is a surface model (it includes vegetation and buildings).

## Cell 0: Google Colab setup (skip this cell on your own computer)
On Colab it connects Google Drive (so your files survive), installs `rasterio` if needed, and asks you to upload the two small GeoJSON files from `data/processed/` on your laptop.

In [ ]:
import os, sys, subprocess
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/kkh-landslide"
    os.makedirs(WORK + "/data/processed", exist_ok=True)
    os.chdir(WORK)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
    needed = ["corridor_10km.geojson", "landslides_su2022_in_corridor.geojson"]
    missing = [n for n in needed if not os.path.exists("data/processed/" + n)]
    if missing:
        print("Please choose these files from your laptop (folder data/processed):", missing)
        up = files.upload()
        for name in up:
            os.replace(name, "data/processed/" + name)
    print("Colab ready. Working folder:", os.getcwd())
else:
    print("Running on your own computer: nothing to set up.")

## Cell 1: imports and settings

In [ ]:
import os, json, math, time
import numpy as np
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

try:
    import rasterio
    from rasterio.warp import reproject, Resampling
    from rasterio.transform import from_origin, rowcol
    from rasterio.features import rasterize
    print("rasterio OK, version", rasterio.__version__)
except Exception as e:
    raise RuntimeError("rasterio could not be loaded: " + str(e)[:200])

CORRIDOR_FILE = "data/processed/corridor_10km.geojson"
POINTS_FILE   = "data/processed/landslides_su2022_in_corridor.geojson"
DEM_DIR       = "data/raw/dem"
BASE_URL      = "https://copernicus-dem-30m.s3.amazonaws.com"
CRS_M         = "EPSG:32643"      # UTM zone 43N, metres
RES_M         = 30                # output cell size in metres

os.makedirs(DEM_DIR, exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
print("Working folder:", os.getcwd())

## Cell 2: which tiles do we need?

In [ ]:
def load_geojson(path):
    with open(path) as f:
        return gpd.GeoDataFrame.from_features(json.load(f)["features"], crs="EPSG:4326")

cor4326 = load_geojson(CORRIDOR_FILE)
cor_geom = cor4326.union_all() if hasattr(cor4326, "union_all") else cor4326.unary_union
minx, miny, maxx, maxy = cor_geom.bounds

tiles = []
for lat in range(math.floor(miny), math.floor(maxy) + 1):
    for lon in range(math.floor(minx), math.floor(maxx) + 1):
        if cor_geom.intersects(box(lon, lat, lon + 1, lat + 1)):
            tiles.append(f"Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM")

print(len(tiles), "tiles needed:")
for t in tiles:
    print("  ", t)

## Cell 3: download the tiles
Each tile is a few tens of MB, so this can take a while. Tiles already on disk are skipped, so you can safely re-run this cell.

In [ ]:
files = []
missing = []
for name in tiles:
    path = os.path.join(DEM_DIR, name + ".tif")
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        print("already have", name)
        files.append(path)
        continue
    url = f"{BASE_URL}/{name}/{name}.tif"
    ok = False
    for attempt in range(3):
        try:
            r = requests.get(url, stream=True, timeout=60)
            if r.status_code == 404:
                print("not available (404):", name)
                missing.append(name)
                break
            r.raise_for_status()
            tmp = path + ".part"
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            os.replace(tmp, path)
            print("downloaded", name, round(os.path.getsize(path) / 1e6, 1), "MB")
            files.append(path)
            ok = True
            break
        except Exception as e:
            print("  attempt", attempt + 1, "failed:", str(e)[:100])
            time.sleep(3)
    if not ok and name not in missing:
        print("giving up on", name, "- run this cell again later")

print()
print("Tiles on disk:", len(files), "| missing:", len(missing))
assert files, "No tiles downloaded. Send me the messages above."

## Cell 4: reproject to metres and cut to the corridor
Each tile is warped straight onto one common metre grid, then everything outside the corridor is set to empty (NaN).

In [ ]:
cor_m = cor4326.to_crs(CRS_M)
geom_m = cor_m.union_all() if hasattr(cor_m, "union_all") else cor_m.unary_union
bx0, by0, bx1, by1 = geom_m.bounds
x0 = math.floor(bx0 / RES_M) * RES_M
y1 = math.ceil(by1 / RES_M) * RES_M
width = int(math.ceil((bx1 - x0) / RES_M))
height = int(math.ceil((y1 - by0) / RES_M))
transform = from_origin(x0, y1, RES_M, RES_M)
print("Output grid:", width, "x", height, "cells of", RES_M, "m")

dem = np.full((height, width), np.nan, dtype="float32")
for f in files:
    with rasterio.open(f) as src:
        tmp = np.full((height, width), np.nan, dtype="float32")
        reproject(source=rasterio.band(src, 1), destination=tmp,
                  src_nodata=src.nodata, dst_transform=transform, dst_crs=CRS_M,
                  dst_nodata=np.nan, resampling=Resampling.bilinear)
        dem = np.where(np.isnan(dem), tmp, dem)
    print("added", os.path.basename(f))

inside = rasterize([(geom_m, 1)], out_shape=(height, width), transform=transform,
                   fill=0, dtype="uint8").astype(bool)
dem[~inside] = np.nan

profile = dict(driver="GTiff", dtype="float32", count=1, crs=CRS_M, transform=transform,
               width=width, height=height, nodata=np.nan, compress="deflate")

def save(name, arr):
    with rasterio.open(f"data/processed/{name}.tif", "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)
    print("saved data/processed/" + name + ".tif")

save("dem_utm", dem)
print("Elevation inside the corridor: min", round(float(np.nanmin(dem))), "m, max", round(float(np.nanmax(dem))), "m")

## Cell 5: slope, aspect and curvature
- **Slope:** steepness in degrees (0 flat, 90 vertical).
- **Aspect:** direction the slope faces, in degrees clockwise from north (0 = north, 90 = east). Flat cells are left empty.
- **Curvature:** a simple measure of how the surface bends. Positive values are valley-like (concave), negative values are ridge-like (convex).

In [ ]:
dy, dx = np.gradient(dem, RES_M)                 # change in height per metre (rows, columns)
slope = np.degrees(np.arctan(np.hypot(dx, dy)))
aspect = np.degrees(np.arctan2(-dx, dy)) % 360
aspect[slope < 0.1] = np.nan
curv = np.gradient(dx, RES_M, axis=1) + np.gradient(dy, RES_M, axis=0)

save("slope", slope)
save("aspect", aspect)
save("curvature", curv)

print("Slope in corridor: mean", round(float(np.nanmean(slope)), 1), "deg, 95th percentile",
      round(float(np.nanpercentile(slope, 95)), 1), "deg")

## Cell 6: sanity check against the landslide points
If the landslides sit on steeper ground than the average corridor pixel, the data and the terrain agree, which is a good sign.

In [ ]:
if not os.path.exists(POINTS_FILE):
    print("Landslide file not found, skipping the check:", POINTS_FILE)
else:
    pts = load_geojson(POINTS_FILE).to_crs(CRS_M)
    rows, cols = rowcol(transform, pts.geometry.x.values, pts.geometry.y.values)
    rows, cols = np.array(rows), np.array(cols)
    ok = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    rows, cols = rows[ok], cols[ok]
    s_pts, e_pts = slope[rows, cols], dem[rows, cols]
    s_all = slope[inside & ~np.isnan(slope)]
    e_all = dem[inside & ~np.isnan(dem)]

    print("Landslide points on the grid:", int(ok.sum()), "of", len(pts))
    print("Median slope  - landslides:", round(float(np.nanmedian(s_pts)), 1),
          "deg | all corridor cells:", round(float(np.median(s_all)), 1), "deg")
    print("Median height - landslides:", round(float(np.nanmedian(e_pts))),
          "m | all corridor cells:", round(float(np.median(e_all))), "m")

    ext = (x0, x0 + width * RES_M, y1 - height * RES_M, y1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 8))
    im = axes[0].imshow(slope[::10, ::10], extent=ext, cmap="viridis", vmin=0, vmax=60)
    axes[0].scatter(pts.geometry.x, pts.geometry.y, s=4, c="red")
    axes[0].set_title("Slope (deg) with landslides in red")
    plt.colorbar(im, ax=axes[0], shrink=0.6)
    axes[1].hist(s_all[::50], bins=40, density=True, alpha=0.5, label="all corridor cells")
    axes[1].hist(s_pts[~np.isnan(s_pts)], bins=40, density=True, alpha=0.5, label="landslide points")
    axes[1].set_xlabel("slope (deg)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()